# Orbit Propagator

Step 1: Load cached ESTCUBE-1 TLE or download once

In [1]:
from pathlib import Path
from urllib.request import urlopen

# ESTCUBE-1 (NORAD ID 39161)
TLE_URL = 'http://celestrak.org/NORAD/elements/gp.php?CATNR=39161&FORMAT=TLE'
TLE_FILE = Path('data/estcube_39161.tle')

def load_or_download_tle() -> str:
    TLE_FILE.parent.mkdir(parents=True, exist_ok=True)

    if TLE_FILE.exists():
        print(f"Using cached file: {TLE_FILE}")
    else:
        print("Downloading ESTCUBE-1 TLE...")
        with urlopen(TLE_URL, timeout=30) as response:
            text = response.read().decode("utf-8", errors="replace")
        TLE_FILE.write_text(text, encoding="utf-8")
        print(f"Saved to: {TLE_FILE}")

    return TLE_FILE.read_text(encoding="utf-8", errors="replace")

tle_text = load_or_download_tle()
print("\nPreview:")
print("\n".join(tle_text.splitlines()[:3]))


Using cached file: data/estcube_39161.tle

Preview:
ESTCUBE 1               
1 39161U 13021C   26112.12892051  .00001110  00000+0  14891-3 0  9995
2 39161  97.7342 168.2770 0008051 187.9450 172.1636 14.81225531696721


Step 2: Extract and print TLE orbital parameters

In [5]:
from datetime import datetime, timedelta, timezone
from math import radians

def tle_epoch_to_datetime(epoch_text: str) -> datetime:
    year = int(epoch_text[:2])
    day_of_year = float(epoch_text[2:])
    full_year = 2000 + year if year < 57 else 1900 + year
    return datetime(full_year, 1, 1, tzinfo=timezone.utc) + timedelta(days=day_of_year - 1)

def parse_tle_parameters(tle_text: str) -> dict[str, float | datetime]:
    lines = [line.strip() for line in tle_text.splitlines() if line.strip()]
    line1 = lines[1]
    line2 = lines[2]

    return {
        "epoch": tle_epoch_to_datetime(line1[18:32]),
        "inclination_rad": radians(float(line2[8:16])),
        "raan_rad": radians(float(line2[17:25])),
        "eccentricity": float(f"0.{line2[26:33].strip()}"),
        "argument_of_periapsis_rad": radians(float(line2[34:42])),
        "mean_anomaly_rad": radians(float(line2[43:51])),
        "mean_motion_rev_per_day": float(line2[52:63]),
    }

tle_parameters = parse_tle_parameters(tle_text)

print("Selected satellite TLE parameters:")
print(f"Epoch: {tle_parameters['epoch'].strftime('%Y-%m-%d %H:%M:%S.%f UTC')}")
print(f"Inclination: {tle_parameters['inclination_rad']:.10f} rad")
print(f"RAAN: {tle_parameters['raan_rad']:.10f} rad")
print(f"Eccentricity: {tle_parameters['eccentricity']:.10f}")
print(f"Argument of periapsis: {tle_parameters['argument_of_periapsis_rad']:.10f} rad")
print(f"Mean anomaly: {tle_parameters['mean_anomaly_rad']:.10f} rad")
print(f"Mean motion: {tle_parameters['mean_motion_rev_per_day']:.8f} rev/day")


Selected satellite TLE parameters:
Epoch: 2026-04-22 03:05:38.732064 UTC
Inclination: 1.7057835818 rad
RAAN: 2.9369877054 rad
Eccentricity: 0.0008051000
Argument of periapsis: 3.2802590627 rad
Mean anomaly: 3.0048216721 rad
Mean motion: 14.81225531 rev/day


Step 3: Propagate the orbit to **2026-04-24 17:00:00**.

In [ ]:
target_time = datetime(2026, 4, 24, 17, 0, 0, tzinfo=timezone.utc)
tle_epoch = tle_parameters["epoch"]

# Propagation time: Dt = t - t0
delta_t = target_time - tle_epoch
delta_t_seconds = delta_t.total_seconds()
delta_t_days = delta_t_seconds / (24 * 60 * 60)

print("Propagation time:")
print(f"t0 (TLE epoch): {tle_epoch.strftime('%Y-%m-%d %H:%M:%S.%f UTC')}")
print(f"t  (target):    {target_time.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"Dt: {delta_t}")
print(f"Dt: {delta_t_seconds:.6f} seconds")
print(f"Dt: {delta_t_days:.10f} solar days")


Propagation time:
t0 (TLE epoch): 2026-04-22 03:05:38.732064 UTC
t  (target):    2026-04-24 17:00:00 UTC
Dt: 2 days, 13:54:21.267936
Dt: 222861.267936 seconds
Dt: 2.5794128233 solar days


In [14]:
from math import degrees, floor

M0_deg = degrees(tle_parameters["mean_anomaly_rad"])
n = tle_parameters["mean_motion_rev_per_day"]

# Mean anomaly at time t:
# M(t) = M0 + 360 * { nDt - INT(nDt) - INT([M0 + 360(nDt - INT(nDt))] / 360) }

completed_orbits = floor(n * delta_t_days)
fractional_orbit = n * delta_t_days - completed_orbits
wrap_count = floor((M0_deg + 360 * fractional_orbit) / 360) # check if fractional part of M(t) exceeds 360 degrees

mean_anomaly_t_deg = M0_deg + 360 * (fractional_orbit - wrap_count)
mean_anomaly_t_rad = radians(mean_anomaly_t_deg)

print("Mean anomaly at target time:")
print(f"M(t): {mean_anomaly_t_rad:.10f} rad")


Mean anomaly at target time:
M(t): 4.3049464755 rad


In [ ]:
from math import cos, sin

# Newton method
def solve_kepler_equation(mean_anomaly_rad: float, eccentricity: float, tolerance: float = 1e-12, max_iterations: int = 50) -> float:
    E = mean_anomaly_rad

    for _ in range(max_iterations):
        residual = E - eccentricity * sin(E) - mean_anomaly_rad
        slope = 1 - eccentricity * cos(E) # dM/dE = 1 - e cos(E)
        correction = residual / slope
        E -= correction

        if abs(correction) < tolerance:
            return E

    raise RuntimeError("Kepler equation solver did not converge")

eccentricity = tle_parameters["eccentricity"]
eccentric_anomaly_t_rad = solve_kepler_equation(mean_anomaly_t_rad, eccentricity)

print("Eccentric anomaly at target time:")
print(f"E(t): {eccentric_anomaly_t_rad:.10f} rad")

Eccentric anomaly at target time:
E(t): 4.3042075192 rad
